# BERT: Pre-training of Deep Bidirectional Transformers - 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)

- Tutorial ID: `expand-bert-paper`
- Tutorial: BERT: Pre-training of Deep Bidirectional Transformers
- Section ID: `expand-bert-paper-code-2`
- Section: 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: BERT로 텍스트 분류 파인튜닝 (SST-2 감정분석)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate
import numpy as np

# ── 1. 데이터 로드 ──
dataset = load_dataset("glue", "sst2")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess(examples):
    return tokenizer(examples['sentence'], truncation=True, max_length=128)

tokenized = dataset.map(preprocess, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ── 2. 모델 로드 (2-class 분류) ──
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)

# ── 3. 평가 메트릭 ──
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

# ── 4. 학습 설정 ──
training_args = TrainingArguments(
    output_dir="./bert-sst2",
    learning_rate=2e-5,                # BERT 권장 LR
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,                # BERT는 2~4 에폭
    weight_decay=0.01,
    warmup_ratio=0.06,                 # Warmup 비율
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# ── 5. 학습 ──
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# ── 6. 추론 ──
text = "This movie was absolutely wonderful and inspiring!"
inputs = tokenizer(text, return_tensors="pt", truncation=True)
with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(logits).item()
    confidence = torch.softmax(logits, dim=-1)[0, pred].item()
    
label_map = {0: "Negative", 1: "Positive"}
print(f"문장: {text}")
print(f"예측: {label_map[pred]} (신뢰도: {confidence:.4f})")